# Graded Open-Ended Lab

## Sequential Data Prediction in Engineering Using Autoregressive (AR) Models and Long Short-Term Memory (LSTM) Networks

**Objective**
This lab explores the application of sequential data prediction in engineering contexts evaluating the performance of both traditional Autoregressive (AR) models and modern Long Short-Term Memory (LSTM) neural networks.

The following section shows the code sample to implement an AR model and an LSTM model.

### Code Sample for AR and LSTM models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

The following code block generate a time series data using the sine function and a normally distributed white noise.

In [ ]:
# Example: Creating a sine wave with noise as the time series data
np.random.seed(42)
time = np.arange(0, 100, 0.1)
data_values = np.sin(time) + np.random.normal(scale=0.5, size=len(time))

# Convert to pandas DataFrame
df = pd.DataFrame({'time': time, 'value': data_values})

# Plot the data
plt.figure(figsize=(10,6))
plt.plot(df['time'], df['value'], label='Original Data')
plt.title('Time Series Data')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.show()

# Split the data into train and test sets
train_size = int(len(df) * 0.8)
train_data = df['value'][:train_size]
test_data = df['value'][train_size:]

### Exploratory Visualizations: ACF and PACF for AR Lag Selection
Before fitting an Autoregressive model, plotting the Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF) helps students analytically determine the optimal lag order instead of guessing.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(train_data, ax=axes[0], lags=30)
axes[0].set_title('Autocorrelation Function (ACF)')
plot_pacf(train_data, ax=axes[1], lags=30, method='ywm')
axes[1].set_title('Partial Autocorrelation Function (PACF)')
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import root_mean_squared_error

The following code block shows the training and evaluation of a autoregressive (AR) model with lags of 5.

In [ ]:
# Fit the Autoregressive (AR) model
model_ar = AutoReg(train_data, lags=5)  # Using lag=5 as an example
model_fit = model_ar.fit()

# Print summary of the model
print(model_fit.summary())

# Make predictions
predictions_ar = model_fit.predict(start=len(train_data), end=len(df)-1, dynamic=False)

# Evaluate the model
rmse = root_mean_squared_error(test_data, predictions_ar)
print(f'RMSE of the AR Model: {rmse}')

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.metrics import mean_squared_error

The following code block shows the data preprocessing prior to LSTM model training specifically:
1. min-max scaling
2. converting the sequential data into chunks of 10
3. splitting the data into training and testing data with 80:20 ratio
4. reshaping the training and testing data to be compatible with LSTM

In [ ]:
# Normalize the data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df['value'].values.reshape(-1,1))

# Create sequences for LSTM
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

sequence_length = 10
X, y = create_sequences(scaled_data, sequence_length)

# Split the data into training and testing sets
train_size_lstm = int(len(X) * 0.8)
X_train, X_test = X[:train_size_lstm], X[train_size_lstm:]
y_train, y_test = y[:train_size_lstm], y[train_size_lstm:]

# Reshape the data to be compatible with LSTM (samples, time steps, features)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

The following code block shows the construction and training of an LSTM network with 50 units of hidden neurons.

In [ ]:
# Build the LSTM model
model = Sequential()
model.add(LSTM(50, input_shape=(X_train.shape[1], 1)))
model.add(Dense(1))  # Output layer
model.compile(optimizer='adam', loss='mse')

# Train the LSTM model
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_data=(X_test, y_test), verbose=0)

# Make predictions
predicted_test = model.predict(X_test)
predicted_test = scaler.inverse_transform(predicted_test)
y_test_rescaled = scaler.inverse_transform(y_test.reshape(-1, 1))

# Evaluate the model
rmse_lstm = root_mean_squared_error(y_test_rescaled, predicted_test)
print(f'RMSE of the LSTM Model: {rmse_lstm}')

### Advanced Comparative Visualizations
To better evaluate micro-level tracking, phase lags, and error distribution between AR and LSTM models, run the visualization cells below.

In [ ]:
# 1. Zoomed-In Comparison Window (First 100 test steps)
zoom_steps = 100
test_time_axis = df['time'][train_size + sequence_length : train_size + sequence_length + zoom_steps]

plt.figure(figsize=(12, 5))
plt.plot(test_time_axis, y_test_rescaled[:zoom_steps], label='Actual Data', color='black', linewidth=2)
plt.plot(test_time_axis, predictions_ar.iloc[:zoom_steps], label='AR Predictions', color='blue', linestyle='--')
plt.plot(test_time_axis, predicted_test[:zoom_steps], label='LSTM Predictions', color='red')
plt.title('Zoomed-In Model Comparison (Phase Lag & Tracking Analysis)')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 2. Residual Error Distribution
ar_residuals = test_data.values[:len(predicted_test)] - predictions_ar.values[:len(predicted_test)]
lstm_residuals = y_test_rescaled.flatten() - predicted_test.flatten()

plt.figure(figsize=(10, 5))
plt.hist(ar_residuals, bins=30, alpha=0.5, label='AR Residuals', color='blue')
plt.hist(lstm_residuals, bins=30, alpha=0.5, label='LSTM Residuals', color='red')
plt.title('Error Residual Distribution Histogram')
plt.xlabel('Prediction Error (Actual - Predicted)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True)
plt.show()

## Instructions

Download the training and testing data by running the following code block. You can view the downloaded files by clicking on "Files" on the vertical toolbar on the left side of this Jupyter notebook.

The original dataset is provided by https://www.kaggle.com/datasets/sumanthvrao/daily-climate-time-series-data.

In [ ]:
!gdown 17kLwUsh0N_bBwmbrvcbOC8L_8I49CyvX
!gdown 18jUJjBe3y3hHxexZfTWORaePrxu7qsAM

Using the training and testing dataset, design and execute the experiments to tune the hyperparameters of the models and compare the performance of AR and LSTM models.

Record the methodology and the results together with discussions in a Jupyter notebook as lab report.

The open-ended lab report will be marked according to the following rubrics.

| Marking Level | Advanced (80–100%) | Proficient (60–79%) | Developing (40–59%) | Needs Improvement (0–39%) |
| :--- | :--- | :--- | :--- | :--- |
| **Implementation & Experimentation (35%)** | Flawless data loading/scaling with strong justification. Rigorous EDA (ACF/PACF) for lag selection. Comprehensive hyperparameter tuning loops with controlled variables. | Correct data loading, scaling, and splitting with basic visual checks. AR and LSTM correctly implemented with standard hyperparameter tuning. | Data loaded and scaled with minimal justification. Lags/windows chosen arbitrarily. Basic model execution with default parameters. | Incomplete data loading or missing scaling. Models fail to execute, have critical errors, or omit required architectures. |
| **Evaluation & Diagnostic Visualizations (25%)** | Evaluated via multi-metric indicators (RMSE, MAE, MAPE). Advanced diagnostics: zoomed-in tracking windows and residual error histograms. | Evaluated via standard metrics (RMSE, MAE). Clear actual-vs-predicted line plots provided for both models with proper labels. | Relies on a single metric without context. Basic global plots that hide micro-errors; poor figure labeling. | Evaluation missing, incomplete, or incorrect. Visualizations absent, broken, or fail to show predictions. |
| **Technical Discussion & Insights (25%)** | Deep critical synthesis comparing linear (AR) vs. non-linear (LSTM) dynamics. Analyzes phase lags, amplitude dampening, and deployment constraints. | Accurately explains performance differences, linking results to linear dependencies vs. deep learning memory cells. | Mostly descriptive discussion restating output values with minimal analysis or theoretical linkage. | Discussion absent or generic. Fails to interpret results, provide comparative evaluation, or show critical thinking. |
| **Code Quality & Notebook Structure (15%)** | Exceptionally clean, modular, and reproducible code with fixed seeds. Professional academic markdown and logical flow. | Organized, error-free code with reasonable comments and clear markdown blocks describing the workflow. | Messy, poorly structured code with minimal comments and disjointed markdown sections. | Unresolved execution errors, completely uncommented code, or disorganized structure missing core documentation. |